# 04 — Choix du modèle — trois algorithmes, un seuil d'automatisation

**Projet** : Routage de tickets de support par un encodeur appris sur le corpus

## Objectifs pédagogiques

1. Comparer les algorithmes de la stack sur le split de validation, à corpus égal.
1. Chiffrer l'effet du rééquilibrage sur la classe minoritaire.
1. Lire la courbe de calibration : ce que « confiance 0,9 » annonce réellement.
1. Arbitrer un seuil d'automatisation sur la calibration, puis le lire sur le test.

## 1. Mise en place

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

# --- Racine du projet ---------------------------------------------------------------------------
# Le notebook s'exécute depuis `notebooks/` : on remonte d'un cran pour pouvoir importer `src`.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from hydra import compose, initialize_config_dir  # noqa: E402
from hydra.core.global_hydra import GlobalHydra  # noqa: E402

from src.schemas.config import validate_config  # noqa: E402
from src.utils.config_access import node  # noqa: E402
from src.utils.logging import setup_logging  # noqa: E402
from src.utils.paths import ProjectPaths  # noqa: E402

# --- Réglages d'affichage -----------------------------------------------------------------------
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 170)
# Le projet journalise au niveau INFO ; un notebook noierait ses tableaux sous ces lignes.
setup_logging(level="WARNING")

# --- Configuration : exactement celle de `python -m src.main` -----------------------------------
# Les notebooks travaillent sur un corpus réduit (320 tickets) quand `data/raw` est
# vide : l'exécution reste rapide. Si le corpus de référence a été généré par `make data`, il est
# utilisé tel quel — les valeurs absolues d'un notebook ne sont pas celles de la référence.
NB_DOCUMENTS = 320

GlobalHydra.instance().clear()
with initialize_config_dir(config_dir=str(PROJECT_ROOT / "conf"), version_base=None):
    CONFIG = validate_config(
        compose(
            config_name="config",
            overrides=[
                "mode=train",
                f"data.n_samples={NB_DOCUMENTS}",
                "seed=42",
                "log_level=WARNING",
                "train.callbacks.progress_bar=false",
            ],
        )
    )

PATHS = ProjectPaths.from_root(PROJECT_ROOT)
# Les notebooks lisent le corpus du projet et écrivent leurs artefacts dans `outputs/notebooks` :
# ils ne doivent jamais écraser ceux produits par `make train` / `make evaluate`.
NB_PATHS = ProjectPaths(
    root=PROJECT_ROOT,
    data_dir=PATHS.data_dir,
    artifacts_dir=PROJECT_ROOT / "outputs" / "notebooks" / "artifacts",
).ensure()

MODEL_NODE = dict(node(CONFIG, "model"))
TEXT_COLUMN = str(MODEL_NODE.get("text_column", "text"))
TARGET_COLUMN = str(CONFIG.data.target or "label")
LABELS = [
    "facturation",
    "livraison",
    "produit_defectueux",
    "remboursement",
    "compte_client",
    "autre",
]
STYLES = [
    "canonique",
    "paraphrase",
    "bruite",
]

print(f"Projet            : {CONFIG.project.name}")
print(f"Tâche             : {CONFIG.metrics.task}")
print(f"Métrique primaire : {CONFIG.metrics.primary} (seuil contractuel : 0.7)")
print(f"Algorithme        : {CONFIG.model.algorithm} ({CONFIG.model.name})")
print(f"Classes           : {len(LABELS)} — {', '.join(LABELS)}")
print(f"Sorties notebook  : {NB_PATHS.artifacts_dir.relative_to(PROJECT_ROOT)}")

In [ ]:
from src.data.generators import SyntheticTicketGenerator
from src.data.loaders import TextLabelLoader

LOADER = TextLabelLoader(
    PATHS,
    formats=CONFIG.data.formats,
    validation_enabled=CONFIG.data.validation.raw,
    lazy_validation=CONFIG.data.validation.lazy,
)
if LOADER.documents_path.exists():
    DOCUMENTS = LOADER.load_documents()
    METADATA = LOADER.load_metadata()
    print(f"Corpus lu depuis data/raw : {len(DOCUMENTS)} tickets, graine {METADATA.get('seed')}")
else:
    # Un clone frais n'a pas encore de données : le notebook reste exécutable. Le corpus réduit est
    # persisté pour que les pipelines (train, evaluate) trouvent les mêmes fichiers que le
    # notebook ; `make data` régénère le corpus de référence.
    bundle = SyntheticTicketGenerator.from_config(CONFIG.data, seed=CONFIG.seed).generate()
    DOCUMENTS, METADATA = bundle.documents, bundle.metadata
    LOADER.save_documents(DOCUMENTS)
    LOADER.save_metadata(METADATA)
    print(f"data/raw absent : corpus de {len(DOCUMENTS)} tickets généré et persisté")

TRAIN = LOADER.split("train", frame=DOCUMENTS)
VALIDATION = LOADER.split("val", frame=DOCUMENTS)
CALIBRATION = LOADER.split("calibration", frame=DOCUMENTS)
TEST = LOADER.split("test", frame=DOCUMENTS)
# La calibration sert à **régler** (seuil d'automatisation), la validation à **choisir**
# (algorithme, rééquilibrage), et le test à **juger** — une seule fois, à la fin.
print(
    "splits :",
    {
        "train": len(TRAIN),
        "val": len(VALIDATION),
        "calibration": len(CALIBRATION),
        "test": len(TEST),
    },
)

LABEL_COUNTS = DOCUMENTS[TARGET_COLUMN].value_counts()
MAJORITY_LABEL = str(LABEL_COUNTS.index[0])
MINORITY_LABEL = str(LABEL_COUNTS.index[-1])
print(
    f"classe majoritaire : {MAJORITY_LABEL} ({LABEL_COUNTS.iloc[0] / len(DOCUMENTS):.1%}) "
    f"| classe minoritaire : {MINORITY_LABEL} ({LABEL_COUNTS.iloc[-1] / len(DOCUMENTS):.1%})"
)

## 2. Les algorithmes de la stack

La stack en déclare trois : moyenne des vecteurs de termes, encodeur à deux couches, encodeur à quatre couches. Un seul est **servi** (celui du manifeste) ; les autres servent de témoins, et c'est leur écart qui donne un sens au choix.

In [ ]:
from src.models import ALGORITHMS, available_algorithms, describe_algorithm

algorithms = available_algorithms(CONFIG.metrics.task)
print("algorithmes disponibles :", algorithms)
documentation = pd.DataFrame([describe_algorithm(name) for name in algorithms])
display(documentation.set_index("name")[["display_name", "rationale", "defaults"]])
print("algorithmes déclarés par le registre de la stack :", sorted(ALGORITHMS))

## 3. Comparaison sur le split de validation

Une seule règle : on **choisit** sur `val`, on ne juge jamais sur `test`. Les modèles sont montés avec les **défauts du registre** — sinon chaque algorithme recevrait les hyper-paramètres d'un autre — et ces défauts incluent le taux d'apprentissage **propre à chaque architecture** (0,02 pour le sac d'embeddings, 0,005 pour les encodeurs, qui divergent au taux du sac). Sur une stack entraînée par époques, seul le **budget d'époques** est réduit ici (`params=__COMPARISON_PARAMS__`) pour tenir dans la durée d'un notebook : un encodeur qui démarre de zéro n'a pas fini d'apprendre en dix époques, et la comparaison le montre — les valeurs du budget du manifeste sont imprimées juste après, à partir des mesures publiées du run de référence. Le tableau mélange qualité (F1 macro, exactitude, kappa), calibration (ECE), coût (latence par ticket) et une lecture métier : le rappel de la classe minoritaire.

In [ ]:
import time

from src.models import build_model
from src.training.metrics import classification_metrics, per_class_frame

validation_truth = VALIDATION[TARGET_COLUMN].astype(str)
rows = []
fitted = {}
for name in algorithms:
    candidate = build_model(CONFIG, algorithm=name, params={"epochs": 10})
    started = time.perf_counter()
    candidate.fit(TRAIN, target=TARGET_COLUMN)
    fit_seconds = time.perf_counter() - started

    started = time.perf_counter()
    probabilities = candidate.predict_proba(VALIDATION[TEXT_COLUMN].astype(str).tolist())
    inference_seconds = time.perf_counter() - started
    predicted = np.asarray(candidate.labels)[probabilities.argmax(axis=1)]
    scores = classification_metrics(
        validation_truth,
        predicted,
        confidences=probabilities.max(axis=1),
        labels=list(candidate.labels),
    )
    per_class = per_class_frame(
        validation_truth, pd.Series(predicted), labels=list(candidate.labels)
    ).set_index("class")
    fitted[name] = candidate
    rows.append(
        {
            "algorithme": name,
            "F1 macro": round(scores["macro_f1"], 4),
            "exactitude": round(scores["accuracy"], 4),
            "kappa": round(scores["cohen_kappa"], 4),
            "ECE": round(scores["expected_calibration_error"], 4),
            f"rappel {MINORITY_LABEL}": round(float(per_class.loc[MINORITY_LABEL, "recall"]), 4),
            "fit (s)": round(fit_seconds, 3),
            "ms / ticket": round(1000.0 * inference_seconds / max(1, len(VALIDATION)), 3),
        }
    )
comparison = pd.DataFrame(rows).set_index("algorithme")
display(comparison)

reference = str(CONFIG.model.algorithm)
print(f"algorithme servi par le manifeste : {reference}")
if {"epochs": 10}:
    print(
        f"budget de la grille : 10 époques par architecture "
        f"(le manifeste en demande {CONFIG.train.epochs} pour le modèle servi)"
    )
for name in comparison.index:
    delta = comparison.loc[name, "F1 macro"] - comparison.loc[reference, "F1 macro"]
    print(
        f"  {name:16s} F1 macro {comparison.loc[name, 'F1 macro']:.4f} ({delta:+.4f} "
        f"contre le modèle servi)"
    )

# Les valeurs du **budget du manifeste** sont publiées par le manifeste lui-même : les lire ici
# évite de conclure d'une grille à budget réduit ce qui vaut au budget de référence.
PUBLISHED_RESULTS = [
    ("F1 macro sur le split de test (six classes, chacune pesant pareil)", "0,7942"),
    ("F1 macro en validation (le split qui a choisi l'architecture)", "0,8022"),
    ("Exactitude sur le split de test (plancher trivial 0,2034)", "0,7853"),
    (
        "Trois architectures, F1 macro en validation (grille de sélection, 719 tickets)",
        "2 couches 0,8022 / sac 0,7455 / 4 couches 0,0559",
    ),
    (
        "Segments paraphrase (52 tickets) / canonique (93) / bruité (32), F1 macro",
        "0,8285 / 0,7892 / 0,7315",
    ),
    (
        "Écart avec la variante `with-tfidf_classifier` (mêmes corpus et splits)",
        "0,7942 contre 0,8408",
    ),
    (
        "Gain de l'encodeur sur les paraphrases, à corpus et splits identiques",
        "+0,0933 (0,8285 contre 0,7352)",
    ),
    ("Kappa de Cohen / exactitude équilibrée", "0,7426 / 0,8069"),
    ("Écart de calibration (confiance moyenne 0,9753 contre exactitude 0,7853)", "0,2099"),
    ("Latence p50 / p95 d'une prédiction (encodeur, 6 classes)", "2,09 ms / 2,22 ms"),
    (
        "Vocabulaire / paramètres / jetons `[UNK]` / tickets tronqués",
        "385 pièces / 344 582 / 0 % / 0 %",
    ),
    ("Ajustement du modèle servi (719 tickets : 10 époques de masquage, 40 d'affinage)", "150 s"),
]
published = [row for row in PUBLISHED_RESULTS if "architecture" in row[0].lower()]
if published:
    print("mesures publiées (budget du manifeste) :", published[0][0])
    print("   ", published[0][1])

**Ce qu'il faut retenir**

- Les trois algorithmes vivent derrière le même contrat : le pipeline ne change pas quand l'algorithme change — c'est ce qui rend la comparaison possible.
- Chaque architecture reçoit le taux d'apprentissage que la stack déclare pour elle : un encodeur initialisé au hasard diverge à celui qui fait converger un sac d'embeddings, et cette grille est un résultat publié, pas un réglage caché.
- Le budget d'entraînement fait partie du résultat : à dix époques, un encodeur n'a pas encore appris ce qu'un sac d'embeddings apprend en une — c'est pourquoi les valeurs de référence du projet sont publiées au budget du manifeste, juste au-dessus, et pas au budget de cette grille.
- Le temps d'ajustement se lit en secondes et la latence en millisecondes par ticket : à ces échelles, le choix se fait sur la qualité, pas sur la vitesse.

## 4. Rééquilibrage : ce que la classe minoritaire coûte

Le manifeste utilise `class_weight='balanced'`. On mesure ce que ce choix apporte — et ce qu'il coûte — en comparant les deux réglages sur le même split, à algorithme égal.

In [ ]:
from src.models import build_model
from src.training.metrics import classification_metrics, per_class_frame

weighting_rows = {}
for label, class_weight in (("balanced", "balanced"), ("sans rééquilibrage", None)):
    candidate = build_model(
        CONFIG, algorithm=reference, params={"estimator": {"class_weight": class_weight}}
    )
    candidate.fit(TRAIN, target=TARGET_COLUMN)
    probabilities = candidate.predict_proba(VALIDATION[TEXT_COLUMN].astype(str).tolist())
    predicted = np.asarray(candidate.labels)[probabilities.argmax(axis=1)]
    scores = classification_metrics(
        validation_truth,
        predicted,
        confidences=probabilities.max(axis=1),
        labels=list(candidate.labels),
    )
    per_class = per_class_frame(
        validation_truth, pd.Series(predicted), labels=list(candidate.labels)
    ).set_index("class")
    weighting_rows[label] = {
        "F1 macro": round(scores["macro_f1"], 4),
        "exactitude": round(scores["accuracy"], 4),
        f"rappel {MINORITY_LABEL}": round(float(per_class.loc[MINORITY_LABEL, "recall"]), 4),
        f"rappel {MAJORITY_LABEL}": round(float(per_class.loc[MAJORITY_LABEL, "recall"]), 4),
    }
weighting = pd.DataFrame(weighting_rows).T
display(weighting)
delta = (
    weighting.loc["balanced", f"rappel {MINORITY_LABEL}"]
    - weighting.loc["sans rééquilibrage", f"rappel {MINORITY_LABEL}"]
)
print(f"écart de rappel sur la classe {MINORITY_LABEL} : {delta:+.4f}")

**Ce qu'il faut retenir**

- Le rééquilibrage déplace du rappel de la classe majoritaire vers la classe rare : c'est exactement ce que demande une métrique macro.
- Sur ce corpus, l'écart est mesuré et **publié** ; sur un corpus où la classe rare serait plus fréquente, le réglage deviendrait moins utile — la question se repose à chaque réentraînement.
- Le même réglage se lit sur les probabilités : un modèle rééquilibré annonce des confiances plus basses, ce qui change le seuil d'automatisation de la section 6.

## 5. Calibration : ce qu'une confiance annonce

Un modèle peut être exact et mal calibré. C'est exactement ce qui décide si l'on peut automatiser un routage : une confiance de 0,9 qui ne vaut que 0,6 envoie les mauvais tickets en production. La courbe est tracée sur la **calibration**, jamais sur le test.

In [ ]:
from src.visualization.plots import plot_calibration

served = fitted[reference]
calibration_scores = served.predict_proba(CALIBRATION[TEXT_COLUMN].astype(str).tolist())
calibration_prediction = np.asarray(served.labels)[calibration_scores.argmax(axis=1)]
calibration_frame = pd.DataFrame(
    {
        "confidence": calibration_scores.max(axis=1),
        "correct": (
            calibration_prediction == CALIBRATION[TARGET_COLUMN].astype(str).to_numpy()
        ).astype(int),
    }
)
CALIBRATION_FIGURE = plot_calibration(
    calibration_frame, NB_PATHS.figures_dir / "04_calibration.png"
)
print(f"figure écrite : {CALIBRATION_FIGURE.relative_to(PROJECT_ROOT)}")
print(
    "confiance moyenne :",
    round(float(calibration_frame["confidence"].mean()), 4),
    "| exactitude observée :",
    round(float(calibration_frame["correct"].mean()), 4),
)

**Ce qu'il faut retenir**

- Si la confiance moyenne dépasse l'exactitude observée, le modèle est **sur-confiant** : il faut alors des seuils plus hauts pour obtenir la même garantie — et l'inverse est une bonne nouvelle.
- La courbe est tracée sur le split de calibration, le seuil y sera arbitré : le test ne sert à aucun réglage.
- La calibration se dégrade quand le corpus est bruité ou quand une classe est rare : deux propriétés que la ventilation par segment du notebook 06 retrouvera.

## 6. Arbitrer un seuil d'automatisation

Règle de service : un ticket est routé automatiquement si la confiance du modèle atteint le seuil ; en dessous, il part en relecture humaine. On cherche le plus petit seuil qui garantit l'exactitude cible sur la calibration, puis on lit le résultat sur le test — une seule fois.

In [ ]:
thresholds = [0.0, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
target_accuracy = 0.95
rows = []
for threshold in thresholds:
    automatic = calibration_frame[calibration_frame["confidence"] >= threshold]
    accuracy = float(automatic["correct"].mean()) if len(automatic) else float("nan")
    rows.append(
        {
            "seuil": threshold,
            "tickets routés": len(automatic),
            "couverture": round(len(automatic) / len(calibration_frame), 4),
            "exactitude des routés": round(accuracy, 4) if len(automatic) else np.nan,
            "tickets en relecture": int(len(calibration_frame) - len(automatic)),
        }
    )
sweep = pd.DataFrame(rows).set_index("seuil")
display(sweep)

# Deux contraintes, pas une : l'exactitude cible **et** une couverture minimale. Un seuil qui
# n'automatise rien est parfait et inutile : on exige de router au moins un ticket sur cinq.
MIN_COVERAGE = 0.2
eligible = sweep[
    (sweep["exactitude des routés"] >= target_accuracy) & (sweep["couverture"] >= MIN_COVERAGE)
]
if len(eligible):
    CHOSEN_THRESHOLD = float(eligible.index.min())
    reason = "plus petit seuil qui satisfait les deux contraintes"
else:
    candidates = sweep[sweep["couverture"] >= MIN_COVERAGE]
    CHOSEN_THRESHOLD = float(candidates["exactitude des routés"].idxmax())
    reason = "aucun seuil n'atteint la cible : meilleur seuil qui route encore des tickets"
print(
    f"règle de service : exactitude cible {target_accuracy:.2f}, couverture minimale "
    f"{MIN_COVERAGE:.0%}"
)
print(f"seuil retenu sur la calibration : {CHOSEN_THRESHOLD:.2f} ({reason})")

In [ ]:
calibration_selection = calibration_frame["confidence"] >= CHOSEN_THRESHOLD
print(
    f"calibration — routage automatique : {int(calibration_selection.sum())}"
    f"/{len(calibration_selection)} tickets "
    f"({float(calibration_selection.mean()):.1%})"
)

test_probabilities = served.predict_proba(TEST[TEXT_COLUMN].astype(str).tolist())
test_predicted = np.asarray(served.labels)[test_probabilities.argmax(axis=1)]
test_confidence = test_probabilities.max(axis=1)
test_truth = TEST[TARGET_COLUMN].astype(str).to_numpy()

mask = test_confidence >= CHOSEN_THRESHOLD
automatic_accuracy = (
    float((test_predicted[mask] == test_truth[mask]).mean()) if mask.any() else float("nan")
)
global_accuracy = float((test_predicted == test_truth).mean())
print(
    f"test — routage automatique : {int(mask.sum())}/{len(mask)} tickets "
    f"({float(mask.mean()):.1%}), exactitude {automatic_accuracy:.4f}"
)
print(f"test — routage complet     : exactitude {global_accuracy:.4f}")
print(
    f"test — relecture humaine   : {int((~mask).sum())} tickets, dont "
    f"{int((test_predicted[~mask] != test_truth[~mask]).sum())} erreurs interceptées"
)

**Ce qu'il faut retenir**

- Le seuil est un arbitrage **métier** : automatiser une partie des tickets à très haute exactitude vaut mieux que d'automatiser tout le stock à l'exactitude moyenne — mais un seuil qui ne route rien est un aveu d'échec, d'où la couverture minimale.
- Le seuil est choisi sur la calibration et lu sur le test : c'est la seule manière d'annoncer une couverture sans se mentir.
- Ce réglage ne remplace pas la métrique contractuelle : la F1 macro reste mesurée sur **tous** les tickets, seuil ou pas.